# 第十一章

## 优化与深度学习
通过最小化损失函数找到最优模型参数，解决深度学习中非凸优化问题。  
损失函数存在大量局部极小值和鞍点。  
使用迭代优化算法来求解，一般只能保证找到局部最小值。  
如果代价函数f是凸的，且限制集合C是凸的，那么就是凸优化问题，局部最小一定是全局最小。 
严格凸优化问题有唯一的全局最小。  
鞍点（saddle point）是指函数的所有梯度都消失但既不是全局最小值也不是局部最小值的任何位置。  
最小化训练误差并不能保证我们找到最佳的参数集来最小化泛化误差。  
优化问题可能有许多局部最小值。  
一个问题可能有很多的鞍点，因为问题通常不是凸的。  
凸：线性回归、softmax回归  
非凸：MLP,CNN,RNN,attention

##  凸性
凸函数(convex function) $f$:给定一个凸集$\mathcal{X}$,如果对于所有$x,x^\prime\in\mathcal{X}$和所有$\lambda\in[0,1]$,函数$f:\mathcal{X}\to\mathbb{R}$是凸的，得到$$\lambda f(x)+(1-\lambda)f(x')\geq f(\lambda x+(1-\lambda)x').$$
特性：凸优化中局部最优解即全局最优解，但深度学习模型多为非凸。
凸集的交点是凸的，并集不是。

## 梯度下降
原理：沿梯度反方向更新参数，公式为 θt+1\=θt−η∇f(θt)，η 为学习率。  
学习率太大会使模型发散，学习率太小会没有进展。 
梯度下降会可能陷入局部极小值，而得不到全局最小值。
缺点：需遍历全量数据，计算开销大，不适用于大规模数据集。

- 挑选一个初始值 $\mathbf{w}_{0}$
- 重复迭代参数 t=1,2,3
$$\mathbf{w}_{t} = \mathbf{w}_{t-1} - \eta \frac{\partial \ell}{\partial \mathbf{w}_{t-1}}$$
- 沿梯度方向将增加损失函数值
- 学习率：步长的超参数

## 随机梯度下降（SGD）
改进：每次随机选取单个样本计算梯度并更新参数，公式为 θt+1\=θt−η∇fi​(θt)。
特点：收敛过程震荡，但可能跳出局部极小值，泛化能力更强。

- 有n个样本时，计算
$$f(\mathbf{x}) = \frac{1}{n} \sum_{i=0}^{n} \ell_i(\mathbf{x})$$ 的导数太贵
- 随机梯度下降在时间 t 随机选项
样本 t_i 来近似 f(x)
$$
\mathbf{x}_t = \mathbf{x}_{t-1} - \eta_t \nabla \ell_{t_i}(\mathbf{x}_{t-1})$$
$$\mathbb{E}\left[\nabla \ell_{t_i}(\mathbf{x})\right] = \mathbb{E}\left[\nabla f(\mathbf{x})\right]$$

## 小批量随机梯度下降（Mini-batch SGD）
折中方案：每次选取小批量数据（如 32/64 样本）计算平均梯度。
优势：兼具 SGD 的效率和梯度下降的稳定性，是深度学习最常用优化方式。
小批量随机梯度下降
- 计算单样本的梯度难完全利用硬件资源
- 小批量随机梯度下降在时间 t 采样一个随机子集 I_t ⊂ {1,...,n} 使得 |I_t| = b
$$\mathbf{x}_t = \mathbf{x}_{t-1} - \frac{\eta_t}{b} \sum_{i \in I_t} \nabla \ell_i(\mathbf{x}_{t-1})$$
- 同样，这是一个无偏的近似，但降低了方差
$$
\mathbb{E}\left[\frac{1}{b} \sum_{i \in I_t} \nabla \ell_i(\mathbf{x})\right] = \nabla f(\mathbf{x})$$


## 动量法（Momentum）
核心思想：引入 “动量” 累积历史梯度，抑制震荡并加速收敛。
公式：   
    -   速度更新：vt\=γvt−1+η∇f(θt)，γ 为动量系数（通常 0.9）。
    -   参数更新：θt+1\=θt−vt。
    
效果：在陡峭方向减小步长，在平缓方向增大步长，类似物理中的惯性。

- 冲量法使用平滑过的梯度对权重更新
$$\mathbf{g}_{t}=\frac{1}{b} \sum_{i \in I_{t}} \nabla \ell_{i}\left(\mathbf{x}_{t-1}\right)$$
$$
\mathbf{v}_{t}=\beta \mathbf{v}_{t-1}+\mathbf{g}_{t} \quad \mathbf{w}_{t}=\mathbf{w}_{t-1}-\eta \mathbf{v}_{t}$$
梯度平滑：$\mathbf{v}_{t}=\mathbf{g}_{t}+\beta \mathbf{g}_{t-1}+\beta^{2} \mathbf{g}_{t-2}+\beta^{3} \mathbf{g}_{t-3}+\ldots$
- $\beta$ 常见取值 [0.5, 0.9, 0.95, 0.99]

## AdaGrad 算法
自适应学习率：对不同参数使用不同学习率，缓解 “鞍点” 和 “稀疏梯度” 问题。
公式：   
    -   累积梯度平方：gt\=gt−1+(∇f(θt))2。
    -   参数更新：θt+1\=θt−gt+ϵ​η​⋅∇f(θt)，ϵ 为平滑项（1e-6）。
    
缺点：学习率单调递减，可能在后期收敛过慢。

## RMSProp 算法
改进 AdaGrad：引入指数加权移动平均，仅保留最近梯度的影响。
公式：    
    -   累积梯度平方：gt\=γgt−1+(1−γ)(∇f(θt))2。
    -   参数更新：θt+1\=θt−gt+ϵ​η​⋅∇f(θt)。
     
效果：避免学习率过早衰减，适用于非平稳目标函数。
 

## Adadelta 算法
无学习率参数：结合 RMSProp 和动量思想，进一步优化自适应学习率。
核心公式：
    
    -   累积梯度平方：E\[g2\]t\=ρE\[g2\]t−1+(1−ρ)(∇f(θt))2。
    -   累积参数更新平方：E\[Δθ2\]t\=ρE\[Δθ2\]t−1+(1−ρ)(Δθt)2。
    -   参数更新：Δθt\=−E\[g2\]t+ϵ​E\[Δθ2\]t−1+ϵ​​⋅∇f(θt)。
 
## Adam 算法
综合优化：结合动量法和 RMSProp，同时跟踪一阶矩（梯度均值）和二阶矩（梯度方差）。
公式：
    
    -   一阶矩：mt\=β1​mt−1+(1−β1​)∇f(θt)。
    -   二阶矩：vt\=β2​vt−1+(1−β2​)(∇f(θt))2。
    -   偏差修正：m^t\=1−β1t​mt​,v^t\=1−β2t​vt​。
    -   参数更新：θt+1\=θt−v^t​+ϵηm^t​。
    
- 默认参数：β1​\=0.9,β2​\=0.999,ϵ\=1e−8，适用于大多数场景。
Adam

- 记录 $\mathbf{v}_t = \beta_1 \mathbf{v}_{t-1} + (1 - \beta_1) \mathbf{g}_t$ 通常 $\beta_1 = 0.9$

- 展开 $\mathbf{v}_t = (1 - \beta_1)(\mathbf{g}_t + \beta_1 \mathbf{g}_{t-1} + \beta_1^2 \mathbf{g}_{t-2} + \beta_1^3 \mathbf{g}_{t-3} + ...)$

- 因为 $\sum_{i=0}^{\infty} \beta_1^i = \frac{1}{1 - \beta_1}$, 所以权重和为1

- 由于 $\mathbf{v}_0 = 0$, 且 $\sum_{i=0}^{t} \beta_1^i = \frac{1 - \beta_1^i}{1 - \beta_1}$,

- 修正 $\hat{\mathbf{v}}_t = \frac{\mathbf{v}_t}{1 - \beta_1^t}$
Adam
- 类似记录 $\mathbf{s}_{t} = \beta_{2} \mathbf{s}_{t-1} + (1 - \beta_{2}) \mathbf{g}_{t}^{2}$, 通常 $\beta_{2} = 0.999$, 且修正 $\hat{\mathbf{s}}_{t} = \frac{\mathbf{s}_{t}}{1 - \beta_{2}^{t}}$
- 计算重新调整后的梯度 $\mathbf{g}_{t}^{\prime} = \frac{\hat{\mathbf{v}}_{t}}{\sqrt{\hat{\mathbf{s}}_{t}} + \epsilon}$
- 最后更新 $\mathbf{w}_{t} = \mathbf{w}_{t-1} - \eta \mathbf{g}_{t}^{\prime}$


## 学习率调度器
动态调整学习率：避免前期收敛慢或后期震荡。
常见策略：
    -   多项式衰减的一种替代方案是乘法衰减，即$\eta_{t+1}\leftarrow\eta_t\cdot\alpha$其中$\alpha\in(0,1)$。为了
防止学习率衰减到一个合理的下界之下，更新方程经常修改为
$$\eta_{t+1}\leftarrow\max(\eta_{\min},\eta_t\cdot\alpha)。$$
    -   保持学习率为一组分段的常量，并且不时地按给定的参数对学习率做乘法衰减。
    -   余弦调度器，函数形式如下所示，学习率的值在$t\in[0,T]$之间。
$$\eta_t=\eta_T+\frac{\eta_0-\eta_T}2(1+\cos(\pi t/T))$$

这里$\eta_0$是初始学习率，$\eta_T$是当$T$时的目标学习率。此外，对于$t>T$,我们只需
将值固定到$\eta_T$而不再增加它。
    -   使用预热期，在此期间学习率将增加至初始最大值，然后冷却直到优化过程结束。
    

* 深度学习模型大多是非凸。
* 小批量随机梯度下降是最常用的优化算法。
* 冲量对梯度做平滑。
* Adam对梯度做平滑，且对梯度各个维度值做重新调整。